>**Projeto Processamento de Big Data** 


>**Grupo 4**: Camila Sousa 111017 | Carolina Brunheta 110888 | Miguel Correia 110786


>**2023/24**

>**Aplicação do Modelo**

# Inicializar sessão

In [1]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from pyspark.ml import PipelineModel, Pipeline


from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("LondonBikeShareUsage") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.maxResultSize", "2048m") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memoryOverhead", "4g") \
    .getOrCreate()

# Importação do ficheiro Parquet

Carregar o modelo e o conjunto de teste 

In [12]:
pipeline_model = PipelineModel.load("notebooks/model-RandomForestRegression")

In [13]:
# Carregar os dados de teste
london_test = spark.read.parquet("notebooks/london_test.parquet")

Testar/Validar o modelo

In [14]:
london_prediction = pipeline_model.transform(london_test)

In [15]:
# Avaliar o modelo com RMSE
evaluator_rmse = RegressionEvaluator(labelCol='duration', predictionCol='prediction', metricName='rmse')
rmse = evaluator_rmse.evaluate(london_prediction)

# Avaliar o modelo com R-squared
evaluator_r2 = RegressionEvaluator(labelCol='duration', predictionCol='prediction', metricName='r2')
r2 = evaluator_r2.evaluate(london_prediction)

from pyspark.sql import functions as F

# Calcular MAPE manualmente
mape = london_prediction.withColumn("abs_error", F.abs(col("duration") - col("prediction")) / col("duration")) \
                        .selectExpr("avg(abs_error) as MAPE") \
                        .collect()[0]["MAPE"]
metrics_data = [
    ("RMSE", rmse),
    ("R2", r2),
    ("MAPE", mape)
]
metrics = spark.createDataFrame(metrics_data, ["Metric", "Value"])
metrics.show()

+------+-------------------+
|Metric|              Value|
+------+-------------------+
|  RMSE|   589.952117434969|
|    R2|0.36199503929444954|
|  MAPE| 0.5247547604280578|
+------+-------------------+



In [16]:
metrics.write.mode("overwrite").csv("./metrics.csv", header=True)